In [1]:
# Kill all processess on GPU
!fuser -v /dev/nvidia* -k

In [2]:
!nvidia-smi

Sun Feb  1 14:50:45 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   71C    P8             12W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [4]:
from pathlib import Path
from huggingface_hub import snapshot_download

data_id = "alxxtexxr/VRBI-Dataset"
allow_dir = 'model_unsloth_Qwen2.5-VL-3B-Instruct.data_jxie_coco_captions_train.v2'

data_dir = snapshot_download(
    repo_id=data_id,
    repo_type='dataset',
    allow_patterns=f'{allow_dir}/**',
    local_dir='data',
    local_dir_use_symlinks=False,
)
vision_data_dir = Path(data_dir) / allow_dir

print("Vision data downloaded to:", vision_data_dir)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:979: UserWarning: `local_dir_use_symlinks` parameter is deprecated and will be ignored. The process to download files to a local folder has been updated and do not rely on symlinks anymore. You only need to pass a destination folder as`local_dir`.
For more details, check out https://huggingface.co/docs/huggingface_hub/main/en/guides/download#download-files-to-local-folder.
  warnings.warn(


Fetching 12 files:   0%|          | 0/12 [00:00<?, ?it/s]

Vision data downloaded to: /content/data/model_unsloth_Qwen2.5-VL-3B-Instruct.data_jxie_coco_captions_train.v2


In [5]:
%%capture
%pip install faiss-gpu-cu12

In [27]:
from pathlib import Path
import numpy as np
import faiss

# Configuration parameters
DATA_DIR = Path('data')
DIM = 2048

K = 4096          # Use 8192 later when you have more data
NITER = 25        # K-means iterations
BATCH_SIZE = 8100 # Streaming batch size
assert BATCH_SIZE % 18**2 == 0, f"BATCH_SIZE ({BATCH_SIZE}) must be divisible by 18^2"
N_EPOCHS = 3      # Number of full passes

SEED = 42

# Set random seeds for reproducibility
np.random.seed(SEED)
# faiss.rand.seed(SEED)

# Load all batch file paths
batch_paths = [f for f in vision_data_dir.glob('*.npz')]

# Initialize KMeans (with GPU support)
kmeans = faiss.Kmeans(
    d=DIM,
    k=K,
    niter=NITER,
    verbose=True,
    gpu=True,
    seed=SEED,
    min_points_per_centroid=10,
    max_points_per_centroid=10000,
)

In [7]:
# Collect initialization data
# KMeans needs good initialization
# We'll sample ~100k vectors

INIT_TARGET = 100_000
INIT_PER_FILE = 15_000

init_buf = []
total = 0

print("Collecting init samples...")

for f in batch_paths:
    vision_data = np.load(f, mmap_mode='r')
    x = vision_data['visual_embs'] # (N, 2048) -> current: (32_400, 2048)

    n = min(INIT_PER_FILE, len(x))
    idx = np.random.choice(len(x), n, replace=False)

    sample = x[idx].astype(np.float32)

    # Normalize
    faiss.normalize_L2(sample)

    init_buf.append(sample)
    total += n

    if total >= INIT_TARGET:
        break

init_data = np.vstack(init_buf)

print("Init data shape:", init_data.shape)

Init data shape: (105000, 2048)


In [8]:
# Initial Training
print("Initial training...")
kmeans.train(init_data)

Initial training...


np.float64(39957.44921875)

In [28]:
# Streaming Training
print("Streaming training...")

for epoch in range(N_EPOCHS):
    print(f"\nEpoch {epoch + 1}/{N_EPOCHS}")

    for f in batch_paths:
        vision_data = np.load(f, mmap_mode='r')
        x = vision_data['visual_embs']

        n = len(x)

        for i in range(0, n, BATCH_SIZE):
            batch = x[i:i + BATCH_SIZE].astype(np.float32)

            # Normalize
            faiss.normalize_L2(batch)

            # One update step
            kmeans.train(batch)

Streaming training...

Epoch 1/3

Epoch 2/3

Epoch 3/3


In [29]:
# Save codebook
codebook = kmeans.centroids

print("Codebook shape:", codebook.shape)

np.save('codebook.npy', codebook)

print("Saved to codebook.npy")

Codebook shape: (4096, 2048)
Saved to codebook.npy


In [30]:
# Build search index (Optional)
# For later quantization

index = faiss.IndexFlatIP(DIM)  # Inner product = cosine
index.add(codebook)

faiss.write_index(index, 'codebook.index')

print("Saved FAISS index")

Saved FAISS index


In [36]:
from huggingface_hub import upload_file

upload_file(
    path_or_fileobj='codebook.npy',
    path_in_repo=allow_dir+'/codebook.npy',
    repo_id='alxxtexxr/VRBI-Dataset',
    repo_type='dataset',
)
upload_file(
    path_or_fileobj='codebook.index',
    path_in_repo=allow_dir+'/codebook.index',
    repo_id='alxxtexxr/VRBI-Dataset',
    repo_type='dataset',
)

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  codebook.npy                :   2%|1         |  552kB / 33.6MB            

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  codebook.index              :  75%|#######4  | 25.1MB / 33.6MB            

CommitInfo(commit_url='https://huggingface.co/datasets/alxxtexxr/VRBI-Dataset/commit/496d2413ccc7e30683da609ebcdabbfaace87d1f', commit_message='Upload model_unsloth_Qwen2.5-VL-3B-Instruct.data_jxie_coco_captions_train.v2/codebook.index with huggingface_hub', commit_description='', oid='496d2413ccc7e30683da609ebcdabbfaace87d1f', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/alxxtexxr/VRBI-Dataset', endpoint='https://huggingface.co', repo_type='dataset', repo_id='alxxtexxr/VRBI-Dataset'), pr_revision=None, pr_num=None)